# ラボ 2: AutoML Tables による赤ちゃんの体重予測モデルのトレーニング

**学習目標**

1. Vertex AI をセットアップする
1. BigQuery からデータセットを作成してインポートする
1. データセットを分析する
1. AutoML Tables モデルをトレーニングする
1. 評価指標を確認する
1. モデルをデプロイする
1. バッチ予測を実行する
1. オンライン予測を実行する


## はじめに 
このノートブックでは、AutoML Tables を使用して、生まれる前の赤ちゃんの体重を予測するモデルをトレーニングします。Vertex AI コンソール UI を使用して BigQuery からトレーニング データセットを作成し、AutoML Tables モデルのトレーニング、評価、予測を行います。

このラボでは、Vertex AI のセットアップ、BigQuery からのデータセットの作成とインポート、データセットの分析、AutoML Tables モデルのトレーニング、トレーニング済みモデルの評価指標の確認、モデルのデプロイ、そして最後にトレーニング済みモデルを使用したバッチ予測とオンライン予測の両方の実行を行います。

各学習目標は、この受講者用ラボ ノートブックで完了する一連のステップに対応しています。

## テーブルが存在することの確認

次のセルを実行して、データセットとデータテーブルが以前に作成されていることを確認します。作成されていない場合は、ラボ [1b_prepare_data_babyweight](../solutions/1b_prepare_data_babyweight.ipynb) に戻って作成してください。

In [ ]:
%load_ext google.cloud.bigquery

In [ ]:
%%bigquery
-- LIMIT 0 is a free query; this allows us to check that the table exists.
SELECT * FROM babyweight.babyweight_augmented_data
LIMIT 0

## ラボ課題 1: Vertex AI のセットアップ

### ステップ 1: Vertex AI を開く
Google Cloud コンソールに移動し、左上隅にあるナビゲーション メニューを開きます。一番下までスクロールして、`人工知能`（Artificial Intelligence）セクションに移動します。`Vertex AI` をクリックして開きます。

<img src="../assets/vertex/01_vertexai_hamburger_dropdown.png" width='70%'>

### ステップ 2: API の有効化
Vertex AI API をまだ有効にしていない場合は、下の画面が表示されます。`VERTEX AI API を有効にする`（ENABLE VERTEX AI API）ボタンをクリックしてください。

<img src="../assets/vertex/02_vertexai_enable_api.png" width='70%'>

## ラボ課題 2: BigQuery からのデータセットの作成とインポート

### ステップ 3: データセット（Datasets）
左側のメニューで `データセット`（Datasets）をクリックします。

<img src="../assets/vertex/03_datasets.png" width='70%'>

Vertex AI のデータセット ページが表示されます。ここにはインポートされたすべてのデータセットが表示されます。赤ちゃんの体重データセットを追加するため、`+ 作成`（+ CREATE）ボタンをクリックします。

<img src="../assets/vertex/04_create_dataset.png" width='70%'>

### ステップ 4: 新しいデータセットの作成
新しいデータセットに一意の名前を付けます。ここでは `babyweight_automl` と名付けましたが、好きな名前を付けて構いません。回帰モデルを構築するため、`表形式`（TABULAR）と `回帰 / 分類`（Regression/classification）を選択します。名前の指定と選択が完了したら、`作成`（CREATE）ボタンをクリックします。

<img src="../assets/vertex/05_dataset_name.png" width='70%'>

### ステップ 5: データのインポート
データセットが作成されたので、Vertex AI と AutoML Tables がトレーニングに使用できるようにデータをインポートしましょう。データは現在すでに BigQuery にあるため、ラジオボタン `BigQuery からテーブルまたはビューを選択`（Select a table or view from BigQuery）を選択します。BigQuery テーブルを検索するためのテキストボックスが表示されます。このフィールドには `project.dataset.table_name` の形式が必要ですが、下に示すように table_name を入力するだけでテーブルを検索できます。入力が完了したら、`続行`（CONTINUE）ボタンをクリックします。

<img src="../assets/vertex/06_select_bigquery_table.png" width='70%'>

<img src="../assets/vertex/07_selected_bigquery_table.png" width='70%'>

## ラボ課題 3: データセットの分析

### ステップ 6: データセットの分析
ここでは `統計情報を生成`（GENERATE STATISTICS）をクリックして、いくつかの基本的な統計情報を分析できます。6 つの特徴量があり、そのうち 4 つが数値、2 つがカテゴリカルであることがわかります。また、すべての列にわたって欠損値がないことも確認できます。これは素晴らしいことです！また、一意の値の数を確認し、予想と比較することもできます。分析に問題がなければ、**新しいモデルをトレーニング**（TRAIN NEW MODEL）をクリックします。（データの品質に十分な自信がある場合は、統計情報の生成を実行せずに、または統計情報の生成が完了する前にモデルのトレーニングを開始することもできます。）

<img src="../assets/vertex/08_generate_statistics.png" width='70%'>

## ラボ課題 4: AutoML Tables モデルのトレーニング

### ステップ 7: モデルタイプの指定とターゲット列の選択
`トレーニング方法`（Training method）セクションで、目的に `回帰`（Regression）を選択し、モデルタイプに `AutoML` を選択して、`続行`（Continue）をクリックします。

<img src="../assets/vertex/09_regression.png" width='70%'>

`モデルの詳細`（Model details）セクションで、ターゲット列として `weight_pounds` を選択し、`続行`（Continue）をクリックします。

<img src="../assets/vertex/10_target_column.png" width='70%'>

### ステップ 8: 入力特徴量の選択
6 つの列をインポートし、そのうちの 1 つである `weight_pounds` はターゲットまたはラベル列として設定しました。これにより、残りは 5 列になります。`トレーニング オプション`（Training options）のドロップダウンには、すべての列のリストが表示されます。4 つの特徴量として `is_male`、`mother_age`、`plurality`、および `gestation_weeks` を使用します。`hashmonth` は、[2_prepare_babyweight](../solutions/2_prepare_babyweight.ipynb) ラボで再現可能なデータ分割を行ったときの残りであり、ここでは不要です。右側のアイコンをクリックして除外してください。

<img src="../assets/vertex/11_feature_selection.png" width='70%'>

### ステップ 9: トレーニングのセットアップ
モデルをトレーニングする準備がほぼ整いました。ここまで多くのステップを踏みましたが、それらは主にデータをインポートし、データが適切であることを確認するためのものでした。周知のとおり、データは機械学習にとって極めて重要であり、データが期待どおりでなければモデルも期待どおりに機能しません。「Garbage in, garbage out（ゴミを入れればゴミが出る）」です。モデルのトレーニングに費やす最大ノード時間である `トレーニング予算`（Training budget）を設定する必要があります。幸いなことに、それより前に改善が見られなくなった場合、（`早期停止を有効にする`（Enable early stopping）トグルがオンになっていれば）トレーニングは停止し、実際に使用したノード時間に対してのみ課金されます。このデータセットの場合、わずか 1 ノード時間の予算で良好な結果が得られました。設定が完了したら、`トレーニングを開始`（START TRAINING）ボタンをクリックします。

<img src="../assets/vertex/12_budget.png" width='70%'>

### ステップ 10: モデルのトレーニングの待機
モデルのトレーニングが開始されました！完了すれば素晴らしいモデルができるはずです。そして、ここからは待機です。データセットのサイズ、トレーニング予算、その他の要因によって、これには数時間から 1 日以上かかる場合があるため、このステップでは気長に待つ必要があります。待っている間におすすめなのは、このシリーズの次のラボを先に進め、トレーニングが完了したらこのラボに戻ってくることです。

<img src="../assets/vertex/13_automl_tables_training.png" width='70%'>

## ラボ課題 5: 評価指標の確認

### ステップ 11: モデルの評価
モデルのトレーニングが完了しました！トレーニング ジョブが完了したというメールを受信したら、上記で見たようにデータセット ページを再度開いて（完了している場合はモデル名をクリック）、または左側のサイドバーで `Vertex AI`、`モデル`（Models）の順に選択し、最近作成されたモデル（`タイプ`（Type）は `表形式`（Tabular）になります）を探してモデルのページに移動できます。

これで `評価`（EVALUATE）タブを確認して、モデルの精度を確認できます。ターゲットが何であったか（`weight_pounds`）、トレーニングが何を最適化対象としたか（`RMSE`）、そして MAE、RMSE などの多くの評価指標が表示されます。わずか 1 時間のトレーニングの後、1.027 の RMSE という素晴らしい結果が得られました！これは AutoML の優れた能力を明確に示しています。下部には特徴量の重要度の棒グラフが表示されます。`gestation_weeks`（妊娠週数）が圧倒的に最も重要ですが、これは一般的に妊娠期間が長いほど赤ちゃんが成長する時間が長く、したがって赤ちゃんの体重が重くなるため、納得のいく結果です。

<img src="../assets/vertex/14_automl_tables_evaluate.png" width='70%'>

## ラボ課題 6: バッチ予測の実行

### ステップ 12: バッチ予測ジョブの作成
新しく作成された AutoML Tables モデルのトレーニングと評価の結果に満足したら、次は機械学習の真骨頂である予測の実行です！モデルが準備できたので、オンライン予測用にモデルをデプロイすることなく、バッチ予測を作成できます。簡単にするために、ここではトレーニングと評価に使用した BigQuery テーブルに対して予測を実行します。これを行うには、`バッチ予測`（BATCH PREDICTIONS）タブを選択し、`バッチ予測を作成`（CREATE BATCH PREDICTION）をクリックして、任意の `バッチ予測名`（Batch prediction name）を指定します。ラジオボタン `BigQuery テーブル`（BigQuery table）を選択し、`BigQuery プロジェクト ID`、`BigQuery データセット ID`、および `BigQuery テーブルまたはビュー ID` を入力します。Cloud Storage の CSV を使用することもできます。次に、結果の出力先を選択する必要があります。出力形式 `BigQuery` を選択し、`Google Cloud プロジェクト ID` を入力します。結果を Cloud Storage に書き出すことも可能です。これらすべてが設定されたら、`作成`（CREATE）をクリックします。これにより、トレーニング済みの AutoML 表形式モデルと上記で選択した場所のデータを使用してバッチ予測ジョブが送信されます。

<img src="../assets/vertex/15_automl_tables_batch_predict.png" width='70%'>

### ステップ 13: バッチ予測ジョブの完了
少し待つと、バッチ予測が完了するはずです。このデータセットでは 30 分もかかりませんでした。先ほどの画面で完了したジョブを確認するか、ナビゲーション メニューから `Vertex AI`、`バッチ予測`（Batch predictions）に移動します。バッチ予測名をクリックします。データの入力元、結果の保存場所、作成日時、処理にかかった時間が表示されます。`エクスポート先`（Export location）リンクをクリックして [BigQuery コンソール UI](https://console.cloud.google.com/bigquery) に移動し、結果を確認します。

<img src="../assets/vertex/18_automl_tables_batch_predict_results.png" width='70%'>

### ステップ 14: バッチ予測データセット
左端のプロジェクト リストに、作業中のプロジェクトが表示されます。矢印をクリックして、プロジェクト内のすべての BigQuery データセットのドロップダウン リストを展開します。前のステップの `結果ディレクトリ`（Results directory）に示されていたものと同じ新しいデータセットが表示されます。そのデータセットのドロップダウンを展開すると、作成された 2 つの BigQuery テーブル `predictions` と `errors` が表示されます。

<img src="../assets/vertex/19_automl_tables_batch_predict_dataset.png" width='70%'>

### ステップ 15: バッチ予測の予測結果
まず `predictions` テーブルを見てみましょう。`プレビュー`（PREVIEW）タブをクリックして内容を表示します。バッチ予測ジョブへの入力データを受け取り、それに 3 つの新しい列を追加した形になっています。モデルの特徴量として使用しなかった列（`hashmonth` など）もそのまま残っていることに注目してください。`prediction_interval` の `start` と `end` の 2 つの列が表示されます。最後の列は予測 `value` であり、行内の対応する特徴量を使用してトレーニング済みの AutoML Tables モデルによって計算された、予測された `weight_pounds` です。

<img src="../assets/vertex/20_automl_tables_batch_predict_prediction_table.png" width='70%'>

### ステップ 16: バッチ予測のエラー
エラーの有無を確認するために `errors` テーブルを見ることもできます。バッチ予測ジョブを実行した際、幸いエラーはありませんでしたが、エラーが発生した場合は間違いなくここを確認します。`errors` テーブルは空だったため、以下にスキーマを示します。ここでも、バッチ予測ジョブへの入力データに 3 つの新しい列が追加された形になっています。保存されたレコードとともに、エラー `code` と `error` メッセージが含まれます。これらは、意図しない動作をデバッグする際に役立ちます。

<img src="../assets/vertex/21_automl_tables_batch_predict_errors_table_schema.png" width='70%'>

## ラボ課題 7: モデルのデプロイ

### ステップ 15: オンライン予測のためのモデルのデプロイ
上記のバッチ ジョブよりも高速な予測を取得したい場合もあります。そのためには、トレーニング済みモデルをデプロイする必要があります。`デプロイとテスト`（DEPLOY & TEST）タブに移動し、`エンドポイントにデプロイ`（DEPLOY TO ENDPOINT）をクリックします。

<img src="../assets/vertex/16_automl_tables_deploy.png" width='70%'>

### ステップ 17: デプロイ設定

エンドポイントに名前を付け、残りの設定はデフォルトのままにして、`デプロイ`（DEPLOY）をクリックします。

<img src="../assets/vertex/17_deploy_options.png" width='70%'>

## ラボ課題 8: オンライン予測の実行

### ステップ 18: オンライン予測のセットアップ
デプロイが完了すると、`エンドポイントにデプロイ` ボタンの下に一覧表示されたエンドポイントが表示され、さらに下にあるモデルの特徴量テーブルを含む `モデルをテスト`（Test your model）エリアがアクティブになります。各特徴量には、列名、データ型、必須かどうか、事前入力された値が表示されます。これらの値はそのままにしておくことも、別の値を入力することもできます。カテゴリカル特徴量の場合は、必ず有効な値を使用してください。そうしないと、語彙外（OOV: out of vocabulary）に割り振られ、トレーニングの成果を十分に活用できなくなります。値の設定が完了したら、`予測`（PREDICT）ボタンをクリックします。

<img src="../assets/vertex/22_automl_tables_before_online_predict.png" width='70%'>

### ステップ 19: オンライン予測の結果
しばらくすると、画面にオンライン予測結果が表示されます。`予測結果`（Prediction result）と `95% 予測区間`（95% prediction interval）が返されます。各特徴量に他の値を試してみて、どのような予測結果になるか確認してみましょう！

<img src="../assets/vertex/23_automl_tables_after_online_predict.png" width='70%'>

## ラボのまとめ: 
このラボでは、Vertex AI での AutoML Tables のセットアップ、BigQuery からのデータセットの作成とインポート、データセットの分析、AutoML Tables モデルのトレーニング、トレーニング済みモデルの評価指標の確認、モデルのデプロイ、およびトレーニング済みモデルを使用したバッチ予測とオンライン予測の両方の実行を行いました。

Copyright 2021 Google LLC
Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at
    https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.